# Cloud reader: Qwen2.5-14B-Instruct

1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. Allow **Google Drive** access when asked (progress is saved there after every batch)
4. First run only: when **Choose Files** appears, pick `cloud/prompts.jsonl`
5. If the runtime disconnects: reconnect and **Run all** again. It resumes where it stopped
6. `answers.json` downloads automatically at the end

In [ ]:
!pip -q install -U bitsandbytes accelerate

In [ ]:
import json, os, re
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
from google.colab import drive, files
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then Run all again"
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/anlp_hw2"
os.makedirs(WORK, exist_ok=True)
PROMPTS, STATE = f"{WORK}/prompts.jsonl", f"{WORK}/state.json"
if not os.path.exists(PROMPTS):
    uploaded = files.upload()
    open(PROMPTS, "wb").write(next(iter(uploaded.values())))
rows = [json.loads(line) for line in open(PROMPTS, encoding="utf-8")]
state = json.load(open(STATE)) if os.path.exists(STATE) else {"answers": {}, "tried": {}}
print(len(rows), "prompts,", len(state["answers"]), "already answered")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
MODEL = "Qwen/Qwen2.5-14B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(
    MODEL, device_map="auto", low_cpu_mem_usage=True, torch_dtype=torch.float16,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=torch.float16))
model.eval()
print("loaded", round(torch.cuda.memory_allocated() / 1e9, 1), "GB")

In [ ]:
from tqdm.auto import tqdm
REFUSALS = re.compile(r"\b(unknown|not (mentioned|provided|specified|available|listed|stated|found|given)|n/a|tbd|tba|"
                      r"no information|cannot be determined|does not (say|mention|contain|specify))\b", re.I)
PRIORITY = {"leaderboard": 0, "dev": 1, "test": 2}

def is_refusal(answer):
    text = answer.strip(" .").lower()
    return not text or text == "none" or bool(REFUSALS.search(text))

@torch.no_grad()
def generate(messages, max_tokens):
    prompts = [tok.apply_chat_template(m, add_generation_prompt=True, tokenize=False) for m in messages]
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [t.strip().split("\n")[0].strip() for t in texts]

def safe_generate(messages, max_tokens):
    try:
        return generate(messages, max_tokens)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(messages) == 1:
            return ["unknown"]
        half = len(messages) // 2
        return safe_generate(messages[:half], max_tokens) + safe_generate(messages[half:], max_tokens)

answers, tried = state["answers"], state["tried"]
for attempt, batch_size in enumerate([8, 4, 16]):
    todo = [r for r in rows if tried.get(r["key"], 0) == attempt
            and (attempt == 0 or is_refusal(answers[r["key"]]))]
    todo.sort(key=lambda r: (PRIORITY[r["key"].split("|")[0]], r["max_tokens"], len(json.dumps(r["attempts"][attempt]))))
    for i in tqdm(range(0, len(todo), batch_size), desc=f"attempt {attempt + 1}: {len(todo)} prompts"):
        batch = todo[i:i + batch_size]
        for r, a in zip(batch, safe_generate([r["attempts"][attempt] for r in batch], max(r["max_tokens"] for r in batch))):
            answers[r["key"]] = a
            tried[r["key"]] = attempt + 1
        json.dump(state, open(STATE, "w"), ensure_ascii=False)
        torch.cuda.empty_cache()
print(len(answers), "answered, refusals left:", sum(is_refusal(a) for a in answers.values()))

In [ ]:
json.dump(answers, open("answers.json", "w"), ensure_ascii=False)
files.download("answers.json")